# C02b v2 — ALISON reproduction on ChatGPT vs human news

This notebook replaces the first C02b notebook, whose parser treated arbitrary integer-leading continuation lines in `Data/ChatGPT.txt` as class labels.

## Reproduction target

ALISON's released `Data/ChatGPT.txt` contains a balanced, paired binary corpus:

- label `0`: the human-written TuringBench news article;
- label `1`: the ChatGPT completion generated from the corresponding article headline.

The release stores each example as a multiline block. Human blocks appear first, followed by the matched ChatGPT blocks in the same order. We retain the released binary labels and assign an explicit `pair_id` by within-class order.

The paper repeats its blind-obfuscation protocol and reports post-ALISON performance for four machine-text detectors. We reproduce the two historical OpenAI GPT-output detectors that remain directly available.

Paper Table 4 targets:

| Detector | Accuracy | F1 | METEOR | USE cosine | BERTScore |
|---|---:|---:|---:|---:|---:|
| GPT Output Detector — Base | 0.5000 | 0.3670 | 0.6966 | 0.8754 | 0.8941 |
| GPT Output Detector — Large | 0.5682 | 0.3623 | 0.6948 | 0.8734 | 0.9017 |

The paper states that only texts **correctly classified before obfuscation** are retained for the reported attack metrics, so the baseline accuracy/F1 on each detector-specific retained subset is 1.0.

## Important data-format correction

`Data/ChatGPT.txt` is not an ordinary one-record-per-line file. A record begins only at a valid class boundary (`0 ` in the human section, `1 ` in the ChatGPT section); all intervening lines belong to that record. Numeric starts on continuation lines (e.g. years or numbered lists) are text, not new labels.

The notebook therefore:

1. parses both released classes as multiline blocks;
2. verifies exactly one human and one ChatGPT record per `pair_id`;
3. samples and splits by `pair_id`, so corresponding topics cannot leak between ALISON's internal-training and attack sets.

## Released-code reconstruction

The public ALISON repository contains several execution defects. We reconstruct the published algorithm from the paper and released code, with explicit compatibility fixes.

Most importantly, the current `NN.py` is dimensionally invalid. An earlier repository revision contains the functional intended architecture (width 512 with 512→512 hidden layers), so v2 reproduces that historical functional architecture rather than inventing a new one.

Primary ALISON settings:

- character + POS n-grams only;
- \(V=\{1,2,3,4\}\);
- up to 256 most frequent features per level and length;
- Integrated Gradients;
- attribution scaling \(c^{\mathrm{length}}\), \(c=1.35\);
- top \(L=15\) POS patterns;
- `bert-large-uncased` phrase replacement;
- output cleaning/lowercasing matching the release for the paper-reproduction condition.

For later RELIC analysis we additionally save a **case-preserved** version of the exact same edits, because global lowercasing is a release-side artifact that could otherwise confound structural measurements.

In [1]:

from pathlib import Path
import os, sys, json, time, gc, math, random, hashlib, subprocess, importlib.util, pickle, re
from collections import Counter

import numpy as np
import pandas as pd
import torch

ROOT = Path("..")
ALISON_DIR = ROOT / "external" / "ALISON"

# NEW DIRECTORY: never reuse the invalid v1 caches.
OUT = ROOT / "results" / "causal" / "C02b_alison_reproduction_v2"
OUT.mkdir(parents=True, exist_ok=True)
ALISON_DIR.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Initial scientifically useful reproduction scale.
# Set None only after the pipeline has been validated.
N_PER_CLASS = 1000
INTERNAL_FRACTION = 0.50

V = [1, 2, 3, 4]
TOP_NGRAMS = 256
TOP_POS_PATTERNS = 15
C_LENGTH = 1.35

# Released filtering is effectively len(pattern) > 1.
MIN_POS_LENGTH_EXCLUSIVE = 1

IG_STEPS = 1024
IG_INTERNAL_BATCH = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("OUT:", OUT)
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 3), "GB")


OUT: ../results/causal/C02b_alison_reproduction_v2
device: cuda
GPU: NVIDIA RTX 2000 Ada Generation Laptop GPU
VRAM: 8.187 GB



## 1. Pin ALISON and check dependencies


In [2]:

REPO_URL = "https://github.com/EricX003/ALISON.git"
ALISON_RELEASE_COMMIT = "ac1fe0fa632e10ab01d0f3a987cf5275b70cd1a7"
CHATGPT_BLOB_SHA = "1a1019459742edef3ef79bbd22c9d3a62e6c714f"

if not ALISON_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ALISON_DIR)], check=True)
    subprocess.run(
        [
            "git", "-C", str(ALISON_DIR),
            "checkout", "--detach", ALISON_RELEASE_COMMIT,
        ],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(ALISON_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

if commit != ALISON_RELEASE_COMMIT:
    raise RuntimeError(
        "Existing ALISON checkout is not at the pinned release commit. "
        f"Expected {ALISON_RELEASE_COMMIT}, got {commit}. "
        "Use a clean checkout at the expected commit before continuing."
    )

chatgpt_blob = subprocess.check_output(
    [
        "git", "-C", str(ALISON_DIR),
        "hash-object", "Data/ChatGPT.txt",
    ],
    text=True,
).strip()

assert chatgpt_blob == CHATGPT_BLOB_SHA, (
    "Data/ChatGPT.txt does not match the audited released artifact."
)

print("ALISON pinned commit:", commit)
print("ChatGPT.txt blob:", chatgpt_blob)

required = {
    "captum": "captum",
    "nltk": "nltk",
    "sklearn": "scikit-learn",
    "transformers": "transformers",
    "bert_score": "bert-score",
    "pyarrow": "pyarrow",
}

missing = [
    pip_name
    for module_name, pip_name in required.items()
    if importlib.util.find_spec(module_name) is None
]

print("missing:", missing)

if missing:
    print("\nInstall in THIS notebook kernel, then restart:")
    print("%pip install " + " ".join(missing))


ALISON pinned commit: ac1fe0fa632e10ab01d0f3a987cf5275b70cd1a7
ChatGPT.txt blob: 1a1019459742edef3ef79bbd22c9d3a62e6c714f
missing: []


In [3]:

import nltk

NLTK_DIR = Path.home() / "nltk_data"
NLTK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["NLTK_DATA"] = str(NLTK_DIR)

if str(NLTK_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DIR))

for pkg in [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "wordnet",
    "omw-1.4",
]:
    try:
        nltk.download(pkg, download_dir=str(NLTK_DIR), quiet=True)
    except Exception as e:
        print("NLTK warning:", pkg, e)

from nltk import pos_tag
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.util import ngrams
from nltk.translate.meteor_score import meteor_score

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from captum.attr import IntegratedGradients
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
)

print("imports OK")


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/user/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


## 2. Correctly parse the released paired binary file

The old notebook's label table containing years and list numbers was the giveaway: arbitrary integer-leading continuation lines were being interpreted as labels.

The actual release has two ordered sections. Human records begin with `0 `; after the final human record, ChatGPT records begin with `1 `. Either record type may continue over many physical lines. The two sections contain the same number of records and preserve corresponding headline order.

In [4]:
CHATGPT_PATH = ALISON_DIR / "Data" / "ChatGPT.txt"
assert CHATGPT_PATH.exists(), CHATGPT_PATH

def _finish_record(rows, label, parts, start_line):
    if label is None:
        return

    text = " ".join(
        piece.strip()
        for piece in parts
        if piece.strip()
    ).strip()

    if not text:
        raise ValueError(
            f"Empty released record for label {label} at line {start_line}"
        )

    pair_id = len(rows[label])
    class_name = "human" if label == 0 else "chatgpt"

    rows[label].append({
        "pair_id": pair_id,
        "source_id": f"{class_name}_{pair_id:05d}",
        "source_line": int(start_line),
        "text": text,
        "label": int(label),
        "class_name": class_name,
    })

def parse_released_chatgpt_binary(path):
    rows = {0: [], 1: []}
    current_label = None
    current_parts = []
    start_line = None
    phase = 0
    ignored_preamble = []
    continuation_numbers = Counter()

    with open(path, "r", encoding="utf-8", errors="replace") as handle:
        for line_no, raw in enumerate(handle):
            line = raw.rstrip("\n\r")

            # Human block boundary. The release places every label-0 block
            # before the first label-1 block.
            if phase == 0 and line.startswith("0 "):
                _finish_record(
                    rows,
                    current_label,
                    current_parts,
                    start_line,
                )
                current_label = 0
                current_parts = [line[2:]]
                start_line = line_no
                continue

            # First 1-marker closes the human section; subsequent 1-markers
            # delimit ChatGPT blocks. Some markers contain the first output
            # fragment and others are followed by a blank line.
            if line.startswith("1 "):
                _finish_record(
                    rows,
                    current_label,
                    current_parts,
                    start_line,
                )
                phase = 1
                current_label = 1
                current_parts = [line[2:]]
                start_line = line_no
                continue

            if current_label is None:
                if line.strip():
                    ignored_preamble.append((line_no, line[:120]))
                continue

            head, sep, _ = line.strip().partition(" ")
            if sep and head.lstrip("+-").isdigit():
                continuation_numbers[head] += 1

            current_parts.append(line)

    _finish_record(
        rows,
        current_label,
        current_parts,
        start_line,
    )

    human = pd.DataFrame(rows[0])
    chatgpt = pd.DataFrame(rows[1])
    corpus = pd.concat([human, chatgpt], ignore_index=True)

    return corpus, human, chatgpt, ignored_preamble, continuation_numbers

corpus, humans, chatgpt, preamble, continuation_numbers = (
    parse_released_chatgpt_binary(CHATGPT_PATH)
)

print("human records:", len(humans))
print("ChatGPT records:", len(chatgpt))
print("ignored preamble lines:", len(preamble))
print("median human words:", int(humans.text.str.split().str.len().median()))
print("median ChatGPT words:", int(chatgpt.text.str.split().str.len().median()))
print("\nMost common integer-leading CONTINUATION lines:")
print(continuation_numbers.most_common(15))

assert len(humans) > 8000
assert len(humans) == len(chatgpt), (
    "Released human and ChatGPT section counts differ; stop and inspect parsing."
)
assert not corpus.source_id.duplicated().any()
assert corpus.text.notna().all()
assert corpus.text.str.strip().ne("").all()
assert set(corpus.label.unique()) == {0, 1}
assert corpus.groupby("pair_id").label.nunique().eq(2).all()
assert corpus.groupby("pair_id").size().eq(2).all()

print("\nFIRST MATCHED PAIR")
print("\nHUMAN\n", humans.text.iloc[0][:1200])
print("\nCHATGPT\n", chatgpt.text.iloc[0][:1200])

human records: 9257
ChatGPT records: 9257
ignored preamble lines: 0
median human words: 305
median ChatGPT words: 228

Most common integer-leading CONTINUATION lines:
[('2020', 3), ('100', 2), ('2014', 2), ('5', 1), ('45', 1), ('1977', 1), ('12', 1), ('26', 1), ('92', 1), ('2015', 1), ('18', 1), ('2013', 1)]

FIRST MATCHED PAIR

HUMAN
 KOCHI: INK, a physical and digital platform for sharing innovative ideas, today announced that Google will continue to support its INK Fellows Programme for the second year running. INK Conference in association with TED is to be held in Kochi this year. The Fellows Programme is one amongst multiple associations between Google and INK. Google's Senior Vice President and Chief Business Officer, Nikesh Arora, will attend as co-host the INK Conference being held here for three days from October 25. Google's support to the Fellows began in 2012, inspired by the Program's goals of encouraging cross disciplinary innovative thinking, creativity and excellence. 

## 3. Create a balanced, pair-aware blind split

We use the release's explicit binary labels:

\[
Y=0:\text{Human},\qquad Y=1:\text{ChatGPT}.
\]

Sampling is deterministic by SHA256 over `pair_id`. Half of the selected pairs become the non-overlapping internal corpus \(X^\*\) used to train ALISON's stylometric classifier; the other half becomes attack/evaluation set \(T\).

Splitting pairs as groups is stricter than an independent row split: the human article and the ChatGPT article generated from its headline always remain on the same side, preventing topic-pair leakage.

The historical OpenAI target detectors are **never used to select features, train ALISON, rank features, or choose replacements**.

In [5]:
pair_ids = pd.Series(sorted(corpus.pair_id.unique()), name="pair_id")

pair_order = pd.DataFrame({"pair_id": pair_ids})
pair_order["_sample_key"] = [
    hashlib.sha256(
        f"{SEED}|sample|{pair_id}".encode("utf-8")
    ).hexdigest()
    for pair_id in pair_order.pair_id
]
pair_order = pair_order.sort_values("_sample_key").reset_index(drop=True)

n_pairs = len(pair_order) if N_PER_CLASS is None else min(
    N_PER_CLASS,
    len(pair_order),
)
pair_order = pair_order.head(n_pairs).copy()

pair_order["_split_key"] = [
    hashlib.sha256(
        f"{SEED}|split|{pair_id}".encode("utf-8")
    ).hexdigest()
    for pair_id in pair_order.pair_id
]
pair_order = pair_order.sort_values("_split_key").reset_index(drop=True)

n_internal_pairs = int(round(n_pairs * INTERNAL_FRACTION))
pair_order["split"] = (
    ["internal_train"] * n_internal_pairs
    + ["attack_test"] * (n_pairs - n_internal_pairs)
)

data = (
    corpus
    .merge(
        pair_order[["pair_id", "split"]],
        on="pair_id",
        how="inner",
        validate="many_to_one",
    )
    .sort_values(["pair_id", "label"])
    .reset_index(drop=True)
)

display(
    data.groupby(["split", "class_name"])
    .size()
    .rename("n")
    .to_frame()
)

assert len(data) == 2 * n_pairs
assert data.groupby(["split", "label"]).size().nunique() == 1
assert data.groupby("pair_id").label.nunique().eq(2).all()
assert data.groupby("pair_id").split.nunique().eq(1).all()
assert not data.source_id.duplicated().any()

SPLIT_PATH = OUT / "binary_split.parquet"
data.to_parquet(SPLIT_PATH, index=False)

print("selected pairs:", n_pairs)
print("saved:", SPLIT_PATH)

n
split          class_name     
attack_test    chatgpt     500
               human       500
internal_train chatgpt     500
               human       500

selected pairs: 1000
saved: ../results/causal/C02b_alison_reproduction_v2/binary_split.parquet



## 4. POS tagging with checkpointing


In [6]:

PENN_TAGS = [
    'CC','CD','DT','EX','FW','IN','JJ','JJR','JJS','LS','MD','NN','NNS','NNP',
    'NNPS','PDT','POS','PRP','PRP$','RB','RBR','RBS','RP','SYM','TO','UH','VB',
    'VBD','VBG','VBN','VBP','VBZ','WDT','WP','WP$','WRB'
]

tag_map = {}
idx = 0
offset = 65
for tag_name in PENN_TAGS:
    if offset + idx == 90:
        offset = 72
    tag_map[tag_name] = chr(offset + idx)
    idx += 1

def tokenize_words(text):
    out = []
    for sent in sent_tokenize(text):
        out.extend(word_tokenize(sent))
    return out

def compressed_pos(text):
    toks = tokenize_words(text)
    return "".join(tag_map.get(tag, tag) for _, tag in pos_tag(toks))

POS_CACHE = OUT / "pos_cache.parquet"

if POS_CACHE.exists():
    pos_df = pd.read_parquet(POS_CACHE)
else:
    pos_df = pd.DataFrame(columns=["source_id","pos_text"])

pos_map = dict(zip(pos_df.source_id.astype(str), pos_df.pos_text.astype(str)))
text_map = dict(zip(data.source_id.astype(str), data.text))

missing_ids = [
    sid for sid in data.source_id.astype(str)
    if sid not in pos_map
]

for sid in tqdm(missing_ids, desc="POS tagging", unit="text"):
    p = compressed_pos(text_map[sid])
    pos_map[sid] = p

    # checkpoint each completed text
    pos_df = pd.concat([
        pos_df,
        pd.DataFrame([{"source_id": sid, "pos_text": p}])
    ], ignore_index=True)

    pos_df.to_parquet(POS_CACHE, index=False)

data["pos_text"] = data.source_id.astype(str).map(pos_map)

assert data.pos_text.notna().all()
assert data.pos_text.str.len().gt(0).all()

print("POS ready:", len(data))


POS tagging: 100%|██████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:15<00:00, 125.13text/s]

POS ready: 2000



## 5. Build ALISON's character + POS feature basis

For each \(n\in\{1,2,3,4\}\), ALISON retains **up to** 256 frequent character and POS n-grams. It is impossible to have 256 distinct POS unigrams (and usually impossible for character unigrams), so v1's assertion that every group must contain exactly 256 features was wrong.

We now assert only:

\[
0 < |\mathcal F_n| \le 256.
\]


In [7]:

def top_ngrams(sequence, n, limit):
    counts = Counter(ngrams(sequence, n))
    return [k for k, _ in counts.most_common(limit)]

internal = data[data.split.eq("internal_train")].copy()

char_total = " ".join(internal.text.tolist())
pos_total = "".join(internal.pos_text.tolist())

char_features = [top_ngrams(char_total, n, TOP_NGRAMS) for n in V]
pos_features = [top_ngrams(pos_total, n, TOP_NGRAMS) for n in V]

feature_audit = pd.DataFrame({
    "n": V,
    "char_features": [len(x) for x in char_features],
    "pos_features": [len(x) for x in pos_features],
})
display(feature_audit)

assert all(0 < len(g) <= TOP_NGRAMS for g in char_features)
assert all(0 < len(g) <= TOP_NGRAMS for g in pos_features)

n_char = sum(map(len, char_features))
n_pos = sum(map(len, pos_features))
n_total = n_char + n_pos

print("char total:", n_char)
print("POS total:", n_pos)
print("feature dimension:", n_total)

with open(OUT/"features.pkl", "wb") as f:
    pickle.dump({
        "V": V,
        "char": char_features,
        "pos": pos_features,
    }, f)

feature_audit.to_csv(OUT/"feature_count_audit.csv", index=False)


,n,char_features,pos_features
0,1,116,43
1,2,256,256
2,3,256,256
3,4,256,256


char total: 884
POS total: 811
feature dimension: 1695



### Release-compatible normalized frequency representation

The public `Utils.py` divides each occurrence count by the number of **distinct** observed n-grams of that length in the document. We preserve that implementation detail for reproduction.


In [8]:

def release_block(sequence, feature_groups):
    vals = []

    for group in feature_groups:
        n = len(group[0])
        denom = len(Counter(ngrams(sequence, n)))

        for feat in group:
            vals.append(
                sequence.count("".join(feat)) / denom
                if denom != 0 else 0.0
            )

    return vals

def alison_rep(text, pos_text):
    return np.asarray(
        release_block(text, char_features)
        + release_block(pos_text, pos_features),
        dtype=np.float32,
    )

REP_PATH = OUT / "feature_matrix.npy"
REP_ROWS = OUT / "feature_matrix_rows.parquet"

can_load = REP_PATH.exists() and REP_ROWS.exists()

if can_load:
    rep_rows = pd.read_parquet(REP_ROWS)
    can_load = (
        rep_rows.source_id.astype(str).tolist()
        == data.source_id.astype(str).tolist()
    )

if can_load:
    X_all = np.load(REP_PATH)
else:
    X_all = np.stack([
        alison_rep(r.text, r.pos_text)
        for r in tqdm(
            data.itertuples(index=False),
            total=len(data),
            desc="ALISON feature vectors",
            unit="text",
        )
    ])

    np.save(REP_PATH, X_all)
    data[["source_id","label","class_name","split"]].to_parquet(REP_ROWS, index=False)

assert X_all.shape == (len(data), n_total)
assert np.isfinite(X_all).all()

print("X:", X_all.shape)


ALISON feature vectors: 100%|███████████████████████████████████████████████████████████████████| 2000/2000 [00:05<00:00, 394.42text/s]

X: (2000, 1695)



## 6. Train ALISON's historical functional internal MLP

The current public `NN.py` is broken because several hidden layers are written as `Linear(in_size, width)` after the first layer.

A **pre-November-2023 revision in the same repository** contains the functional architecture: width 512 with 512→512 hidden layers, dropout 0.40, then 512→128→classes. We use that historical released architecture.

The internal corpus is further split 85/15 for train/validation, preserving the attack set as completely untouched.


In [9]:

class HistoricalAlisonMLP(nn.Module):
    def __init__(self, in_size, num_classes):
        super().__init__()
        width = 512
        dp = 0.40

        layers = [
            nn.Dropout(dp),
            nn.Linear(in_size, width),
            nn.ReLU(),
        ]

        # Historical functional NN.py contains seven 512 -> 512 hidden blocks.
        for _ in range(7):
            layers += [
                nn.Dropout(dp),
                nn.Linear(width, width),
                nn.ReLU(),
            ]

        layers += [
            nn.Dropout(dp),
            nn.Linear(width, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        ]

        self.stack = nn.Sequential(*layers)

    def forward(self, x):
        return self.stack(x)

internal_idx = np.where(data.split.eq("internal_train").to_numpy())[0]
attack_idx = np.where(data.split.eq("attack_test").to_numpy())[0]

train_i, val_i = train_test_split(
    internal_idx,
    test_size=0.15,
    random_state=1,
    shuffle=True,
    stratify=data.iloc[internal_idx].label.to_numpy(),
)

scaler = StandardScaler().fit(X_all[train_i])

X_scaled = scaler.transform(X_all).astype(np.float32)
y_all = data.label.to_numpy(np.int64)

with open(OUT/"scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

Xtr = torch.tensor(X_scaled[train_i], dtype=torch.float32)
ytr = torch.tensor(y_all[train_i], dtype=torch.long)
Xva = torch.tensor(X_scaled[val_i], dtype=torch.float32)
yva = torch.tensor(y_all[val_i], dtype=torch.long)

MODEL_PATH = OUT / "internal_mlp_historical.pt"

internal_model = HistoricalAlisonMLP(n_total, 2).to(DEVICE)

if MODEL_PATH.exists():
    internal_model.load_state_dict(
        torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
    )
    print("loaded cached internal model")

else:
    loader = DataLoader(
        TensorDataset(Xtr, ytr),
        batch_size=512,
        shuffle=True,
    )

    optimizer = torch.optim.Adam(
        internal_model.parameters(),
        lr=0.01,
        weight_decay=0.0,
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.30,
    )

    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(1, 31):
        internal_model.train()
        correct = total = 0
        losses = []

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = internal_model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)

        scheduler.step()

        if epoch == 1 or epoch % 5 == 0:
            internal_model.eval()
            with torch.no_grad():
                val_pred = (
                    internal_model(Xva.to(DEVICE))
                    .argmax(1)
                    .cpu()
                    .numpy()
                )

            val_acc = accuracy_score(yva.numpy(), val_pred)

            print(
                f"epoch {epoch:02d} | "
                f"loss {np.mean(losses):.4f} | "
                f"train {correct/total:.4f} | "
                f"val {val_acc:.4f}"
            )

    torch.save(internal_model.state_dict(), MODEL_PATH)

internal_model.eval()

with torch.no_grad():
    val_pred = internal_model(Xva.to(DEVICE)).argmax(1).cpu().numpy()
    attack_pred = (
        internal_model(
            torch.tensor(X_scaled[attack_idx], device=DEVICE)
        )
        .argmax(1)
        .cpu()
        .numpy()
    )

internal_val_acc = accuracy_score(yva.numpy(), val_pred)
internal_attack_acc = accuracy_score(y_all[attack_idx], attack_pred)

print("\ninternal validation accuracy:", internal_val_acc)
print("internal attack-set accuracy:", internal_attack_acc)

with open(OUT/"internal_classifier_eval.json", "w") as f:
    json.dump({
        "validation_accuracy": float(internal_val_acc),
        "attack_accuracy": float(internal_attack_acc),
        "historical_architecture_ref": "e47d009eda89bd42a13d49c32ebc9c1e56b4d569",
    }, f, indent=2)


epoch 01 | loss 1.5782 | train 0.5094 | val 0.5000
epoch 05 | loss 0.4934 | train 0.5000 | val 0.5000
epoch 10 | loss 0.3872 | train 0.9494 | val 0.9733
epoch 15 | loss 0.3445 | train 0.9624 | val 0.9733
epoch 20 | loss 0.3517 | train 0.9600 | val 0.9733
epoch 25 | loss 0.3456 | train 0.9635 | val 0.9733
epoch 30 | loss 0.3550 | train 0.9529 | val 0.9733

internal validation accuracy: 0.9733333333333334
internal attack-set accuracy: 0.967



## 7. Freeze the external detector evaluation subsets

The paper retains only samples each target detector classified correctly **before** obfuscation. We therefore score the untouched attack set now and save those masks.

These detector outputs are **not passed to ALISON**. They define evaluation subsets only.


In [11]:

ATTACK = data.iloc[attack_idx][
    ["source_id","text","label","class_name"]
].reset_index(drop=True)

DETECTORS = {
    "openai_base": "openai-community/roberta-base-openai-detector",
    "openai_large": "openai-community/roberta-large-openai-detector",
}

BASELINE_PATH = OUT / "external_detector_baseline.parquet"

if BASELINE_PATH.exists():
    baseline_scores = pd.read_parquet(BASELINE_PATH)

else:
    all_rows = []

    # Temporarily move tiny internal model off GPU.
    internal_model = internal_model.cpu()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for detector_name, checkpoint in DETECTORS.items():
        print("\n==", detector_name, "==")

        tok = AutoTokenizer.from_pretrained(checkpoint)
        clf = AutoModelForSequenceClassification.from_pretrained(
            checkpoint
        ).to(DEVICE).eval()

        id2label = {
            int(k): str(v)
            for k, v in clf.config.id2label.items()
        }

        fake_matches = [
            k
            for k, v in id2label.items()
            if "fake" in v.lower()
        ]
        
        if fake_matches:
            fake_id = fake_matches[0]
        else:
            assert checkpoint in {
                "openai-community/roberta-base-openai-detector",
                "openai-community/roberta-large-openai-detector",
            }
            assert clf.config.num_labels == 2
        
            fake_id = 0
        
            print(
                "Generic checkpoint labels:",
                id2label,
                "| using class 0 = Fake/generated",
            )

        probs = []

        with torch.inference_mode():
            for start in tqdm(
                range(0, len(ATTACK), 16),
                desc=detector_name,
                unit="batch",
            ):
                batch = ATTACK.text.iloc[start:start+16].tolist()

                enc = tok(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(DEVICE)

                p = (
                    clf(**enc)
                    .logits
                    .softmax(-1)[:, fake_id]
                    .cpu()
                    .numpy()
                )

                probs.extend(p.tolist())

        probs = np.asarray(probs)
        preds = (probs >= 0.5).astype(int)
        y = ATTACK.label.to_numpy()

        print("original accuracy:", accuracy_score(y, preds))
        print("retained:", int((preds == y).sum()), "/", len(y))

        for j, r in ATTACK.iterrows():
            all_rows.append({
                "source_id": r.source_id,
                "detector": detector_name,
                "label": int(r.label),
                "class_name": r.class_name,
                "original_fake_prob": float(probs[j]),
                "original_pred": int(preds[j]),
                "original_correct": bool(preds[j] == y[j]),
            })

        del clf, tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    baseline_scores = pd.DataFrame(all_rows)
    baseline_scores.to_parquet(BASELINE_PATH, index=False)

display(
    baseline_scores
    .groupby("detector")
    .agg(
        n=("source_id","size"),
        retained=("original_correct","sum"),
    )
)

# Restore internal model for IG.
internal_model = HistoricalAlisonMLP(n_total, 2)
internal_model.load_state_dict(
    torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
)
internal_model = internal_model.to(DEVICE).eval()



== openai_base ==


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 9232.89it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
openai_base: 100%|██████████████████████████████████████████████████████████████████████████████████| 63/63 [00:25<00:00,  2.48batch/s]


original accuracy: 0.965
retained: 965 / 1000

== openai_large ==


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 393/393 [00:00<00:00, 10942.97it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-large-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generic checkpoint labels: {0: 'LABEL_0', 1: 'LABEL_1'} | using class 0 = Fake/generated


openai_large: 100%|█████████████████████████████████████████████████████████████████████████████████| 63/63 [01:22<00:00,  1.31s/batch]


original accuracy: 0.882
retained: 882 / 1000


,n,retained
detector,,
openai_base,1000,965
openai_large,1000,882



## 8. Integrated Gradients ranking

For each attack text, compute IG toward its **true Human/ChatGPT class**, multiply each POS feature attribution by \(1.35^{n}\), rank positive contributions descending, and retain the top 15 POS patterns of length at least 2.

The target detector plays no role here.


In [12]:

flat_pos = [
    feat
    for group in pos_features
    for feat in group
]

assert len(flat_pos) == n_pos

RANK_PATH = OUT / "ig_ranked_pos.parquet"

if RANK_PATH.exists():
    ranks = pd.read_parquet(RANK_PATH)
else:
    ranks = pd.DataFrame(columns=[
        "source_id",
        "label",
        "patterns_json",
        "scores_json",
        "seconds",
    ])

done = set(ranks.source_id.astype(str)) if len(ranks) else set()

ig = IntegratedGradients(internal_model)

jobs = [
    i for i in attack_idx
    if str(data.iloc[i].source_id) not in done
]

for i in tqdm(jobs, desc="Integrated Gradients", unit="text"):
    r = data.iloc[i]

    x = torch.tensor(
        X_scaled[i:i+1],
        dtype=torch.float32,
        device=DEVICE,
    )

    t0 = time.time()

    attr = (
        ig.attribute(
            x,
            target=int(r.label),
            n_steps=IG_STEPS,
            internal_batch_size=IG_INTERNAL_BATCH,
        )[0]
        .detach()
        .cpu()
        .numpy()
    )

    pos_attr = attr[n_char:n_char+n_pos]

    weighted = np.asarray([
        pos_attr[j] * (C_LENGTH ** len(flat_pos[j]))
        for j in range(n_pos)
    ])

    order = np.argsort(weighted)[::-1]

    selected = [
        j for j in order
        if len(flat_pos[j]) > MIN_POS_LENGTH_EXCLUSIVE
    ][:TOP_POS_PATTERNS]

    patterns = [
        "".join(flat_pos[j])
        for j in selected
    ]

    scores = [
        float(weighted[j])
        for j in selected
    ]

    ranks = pd.concat([
        ranks,
        pd.DataFrame([{
            "source_id": str(r.source_id),
            "label": int(r.label),
            "patterns_json": json.dumps(patterns),
            "scores_json": json.dumps(scores),
            "seconds": time.time() - t0,
        }])
    ], ignore_index=True)

    ranks.to_parquet(RANK_PATH, index=False)
    done.add(str(r.source_id))

print("ranked:", len(ranks), "/", len(attack_idx))
print("mean IG sec/text:", ranks.seconds.astype(float).mean())


Integrated Gradients: 100%|██████████████████████████████████████████████████████████████████████| 1000/1000 [08:24<00:00,  1.98text/s]

ranked: 1000 / 1000
mean IG sec/text: 0.49752848625183105



## 9. BERT-large phrase replacement

The public replacer inserts `[MASK]` at NLTK-word positions but then accidentally indexes BERT subword logits with those same word indices. That is not the algorithm described in the paper.

v2 implements the paper's intended operation:

1. locate the selected POS phrase in NLTK word space;
2. replace that phrase by one `[MASK]` token per word;
3. tokenize a local context window with BERT;
4. identify the **actual BERT mask-token positions**;
5. fill all masks from one forward pass.

All non-overlapping matches of the top patterns are processed, matching ALISON's described greedy procedure.

Two outputs are saved:

- `obfuscated_release`: release-style cleaning + global lowercasing, used for Table-4 reproduction;
- `obfuscated_case_preserved`: identical BERT edits without global lowercasing, saved for later RELIC collateral-effect analysis.


In [13]:

# Internal model no longer needed on GPU.
del internal_model, ig
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

MLM_ID = "bert-large-uncased"

mlm_tok = AutoTokenizer.from_pretrained(MLM_ID)
mlm = AutoModelForMaskedLM.from_pretrained(MLM_ID).to(DEVICE).eval()

MASK = mlm_tok.mask_token

def compressed_tags_for_words(words):
    return [
        tag_map.get(tag, tag)
        for _, tag in pos_tag(words)
    ]

def clean_spacing(text, lowercase=False):
    text = text.strip().strip('"').strip("'")
    text = re.sub(r"\s+", " ", text)

    for punct in ["!", ".", "?", ":", ";", ","]:
        text = text.replace(f" {punct}", punct)

    text = text.replace("' ", "'")
    text = text.replace("#", "")
    text = text.replace("( ", "(").replace(" )", ")")

    return text.lower() if lowercase else text

def local_window(words, lo, hi, max_tokens=500):
    """
    Return a word window containing [lo:hi] whose BERT tokenization fits.
    This guarantees the masked phrase is not lost to 512-token truncation.
    """
    left = max(0, lo - 180)
    right = min(len(words), hi + 180)

    while True:
        test_words = list(words[left:right])
        local_lo = lo - left
        local_hi = hi - left
        test_words[local_lo:local_hi] = [MASK] * (hi - lo)

        n_tok = len(
            mlm_tok(
                " ".join(test_words),
                add_special_tokens=True,
            )["input_ids"]
        )

        if n_tok <= max_tokens:
            return left, right

        # Shrink whichever side has more spare context.
        left_spare = lo - left
        right_spare = right - hi

        if left_spare >= right_spare and left < lo:
            left += max(1, min(20, left_spare))
        elif right > hi:
            right -= max(1, min(20, right_spare))
        else:
            raise RuntimeError("Masked span itself is too large for BERT.")

@torch.inference_mode()
def replace_interval(words, lo, hi):
    original_span = list(words[lo:hi])

    left, right = local_window(words, lo, hi)

    local = list(words[left:right])
    local_lo = lo - left
    local_hi = hi - left

    local[local_lo:local_hi] = [MASK] * (hi - lo)

    enc = mlm_tok(
        " ".join(local),
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    mask_positions = (
        enc["input_ids"][0]
        .eq(mlm_tok.mask_token_id)
        .nonzero(as_tuple=False)
        .flatten()
    )

    expected_masks = hi - lo

    if len(mask_positions) != expected_masks:
        # Safe fallback: preserve original phrase rather than silently corrupting it.
        return words, 0, 0, False

    logits = mlm(**enc).logits[0]
    pred_ids = logits[mask_positions].argmax(-1).tolist()
    pred_tokens = mlm_tok.convert_ids_to_tokens(pred_ids)

    replacements = []

    for original_word, pred in zip(original_span, pred_tokens):
        pred = pred.replace("##", "")
        pred = re.sub(r"[^\w\s'-]", "", pred).strip()

        replacements.append(
            pred if pred else original_word
        )

    actual_changes = sum(
        str(old).casefold() != str(new).casefold()
        for old, new in zip(original_span, replacements)
    )

    new_words = list(words)
    new_words[lo:hi] = replacements

    return new_words, len(replacements), actual_changes, True

rank_map = {
    str(r.source_id): json.loads(r.patterns_json)
    for r in ranks.itertuples(index=False)
}

OBF_PATH = OUT / "alison_outputs.parquet"

if OBF_PATH.exists():
    obf = pd.read_parquet(OBF_PATH)
else:
    obf = pd.DataFrame(columns=[
        "source_id",
        "label",
        "class_name",
        "original",
        "obfuscated_release",
        "obfuscated_case_preserved",
        "n_candidate_intervals",
        "n_changed_intervals",
        "n_replaced_tokens",
        "n_lexically_changed_tokens",
        "masked_word_fraction",
        "lexical_change_fraction",
        "word_change_fraction",
        "replacement_failures",
        "seconds",
    ])

done_obf = set(obf.source_id.astype(str)) if len(obf) else set()

attack_rows = data[data.split.eq("attack_test")].copy()

jobs = [
    r
    for r in attack_rows.itertuples(index=False)
    if str(r.source_id) not in done_obf
]

for r in tqdm(jobs, desc="ALISON BERT replacement", unit="text"):
    t0 = time.time()

    words = tokenize_words(r.text)
    tags_now = compressed_tags_for_words(words)

    intervals = []

    for pattern in rank_map[str(r.source_id)]:
        pat = list(pattern)
        n = len(pat)

        for start in range(0, max(0, len(tags_now) - n + 1)):
            if tags_now[start:start+n] == pat:
                intervals.append((start, start+n))

    changed = [False] * len(words)

    changed_intervals = 0
    replaced_tokens = 0
    lexically_changed_tokens = 0
    failures = 0

    for lo, hi in intervals:
        if hi > len(changed):
            continue

        if any(changed[lo:hi]):
            continue

        new_words, n_replaced, n_lexically_changed, success = replace_interval(
            words,
            lo,
            hi,
        )

        if success:
            words = new_words
            changed[lo:hi] = [True] * (hi-lo)
            changed_intervals += 1
            replaced_tokens += n_replaced
            lexically_changed_tokens += n_lexically_changed
        else:
            failures += 1

    case_preserved = clean_spacing(
        " ".join(words),
        lowercase=False,
    )

    release_text = clean_spacing(
        " ".join(words),
        lowercase=True,
    )

    row = pd.DataFrame([{
        "source_id": str(r.source_id),
        "label": int(r.label),
        "class_name": str(r.class_name),
        "original": r.text,
        "obfuscated_release": release_text,
        "obfuscated_case_preserved": case_preserved,
        "n_candidate_intervals": int(len(intervals)),
        "n_changed_intervals": int(changed_intervals),
        "n_replaced_tokens": int(replaced_tokens),
        "n_lexically_changed_tokens": int(lexically_changed_tokens),
        "masked_word_fraction": float(sum(changed) / max(1, len(changed))),
        "lexical_change_fraction": float(lexically_changed_tokens / max(1, len(changed))),
        # Backward-compatible alias: this now means actual lexical changes.
        "word_change_fraction": float(lexically_changed_tokens / max(1, len(changed))),
        "replacement_failures": int(failures),
        "seconds": float(time.time() - t0),
    }])

    obf = pd.concat([obf, row], ignore_index=True)

    # Checkpoint EVERY text.
    obf.to_parquet(OBF_PATH, index=False)
    done_obf.add(str(r.source_id))

print("saved:", len(obf), "/", len(attack_rows))
print("mean sec/text:", obf.seconds.astype(float).mean())

display(
    obf[
        [
            "n_candidate_intervals",
            "n_changed_intervals",
            "n_replaced_tokens",
            "n_lexically_changed_tokens",
            "masked_word_fraction",
            "lexical_change_fraction",
            "word_change_fraction",
            "replacement_failures",
            "seconds",
        ]
    ].describe()
)


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 394/394 [00:00<00:00, 10291.81it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-large-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
ALISON BERT replacement: 100%|███████████████████████████████████████████████████████████████████| 1000/1000 [24:54<00:00,  1.49s/text]

saved: 1000 / 1000
mean sec/text: 1.4788684418201448


,n_candidate_intervals,n_changed_intervals,n_replaced_tokens,n_lexically_changed_tokens,masked_word_fraction,lexical_change_fraction,word_change_fraction,replacement_failures,seconds
count,1000,1000,1000,1000,1000.000000,1000.000000,1000.000000,1000,1000.000000
unique,99,47,111,78,886.000000,856.000000,856.000000,1,1000.000000
top,32,19,67,35,0.380814,0.168605,0.168605,0,0.630591
freq,44,71,33,48,18.000000,19.000000,19.000000,1000,1.000000



## 10. Post-ALISON detector reproduction

For each historical OpenAI detector:

1. take only its own `original_correct == True` subset;
2. baseline accuracy/F1 is therefore exactly 1.0;
3. evaluate the release-compatible ALISON output;
4. compare against Table 4.

This is the paper's attack estimand, not a newly calibrated detector experiment.


In [14]:

# Free BERT MLM before RoBERTa detectors.
del mlm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

merged = (
    ATTACK
    .merge(
        obf[
            [
                "source_id",
                "obfuscated_release",
                "obfuscated_case_preserved",
                "word_change_fraction",
                "seconds",
            ]
        ],
        on="source_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(merged) == len(ATTACK)

PAPER_TARGETS = {
    "openai_base": {
        "accuracy": 0.5000,
        "f1": 0.3670,
        "meteor": 0.6966,
        "bertscore": 0.8941,
    },
    "openai_large": {
        "accuracy": 0.5682,
        "f1": 0.3623,
        "meteor": 0.6948,
        "bertscore": 0.9017,
    },
}

summary_rows = []
post_rows = []

for detector_name, checkpoint in DETECTORS.items():
    print("\n==", detector_name, "==")

    tok = AutoTokenizer.from_pretrained(checkpoint)
    clf = AutoModelForSequenceClassification.from_pretrained(
        checkpoint
    ).to(DEVICE).eval()

    id2label = {
        int(k): str(v)
        for k, v in clf.config.id2label.items()
    }

    fake_matches = [
        k
        for k, v in id2label.items()
        if "fake" in v.lower()
    ]
    
    if fake_matches:
        fake_id = fake_matches[0]
    else:
        assert checkpoint in {
            "openai-community/roberta-base-openai-detector",
            "openai-community/roberta-large-openai-detector",
        }
        assert clf.config.num_labels == 2
    
        fake_id = 0
    
        print(
            "Generic checkpoint labels:",
            id2label,
            "| using class 0 = Fake/generated",
        )

    base = baseline_scores[
        baseline_scores.detector.eq(detector_name)
    ][["source_id","original_correct"]].copy()

    eval_df = merged.merge(
        base,
        on="source_id",
        how="inner",
        validate="one_to_one",
    )

    eval_df = eval_df[
        eval_df.original_correct
    ].reset_index(drop=True)

    probs = []

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(eval_df), 16),
            desc=f"{detector_name} post-ALISON",
            unit="batch",
        ):
            batch = eval_df.obfuscated_release.iloc[
                start:start+16
            ].tolist()

            enc = tok(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(DEVICE)

            p = (
                clf(**enc)
                .logits
                .softmax(-1)[:, fake_id]
                .cpu()
                .numpy()
            )

            probs.extend(p.tolist())

    probs = np.asarray(probs)
    pred = (probs >= 0.5).astype(int)
    y = eval_df.label.to_numpy(int)

    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred, zero_division=0)

    target = PAPER_TARGETS[detector_name]

    print("retained n:", len(y))
    print("post accuracy:", acc, "| paper:", target["accuracy"])
    print("post F1:", f1, "| paper:", target["f1"])

    summary_rows.append({
        "detector": detector_name,
        "n_retained": len(y),
        "baseline_accuracy": 1.0,
        "baseline_f1": 1.0,
        "post_accuracy": acc,
        "paper_accuracy": target["accuracy"],
        "post_f1": f1,
        "paper_f1": target["f1"],
        "accuracy_abs_error": abs(acc-target["accuracy"]),
        "f1_abs_error": abs(f1-target["f1"]),
    })

    for j, r in eval_df.iterrows():
        post_rows.append({
            "source_id": r.source_id,
            "detector": detector_name,
            "label": int(r.label),
            "post_fake_prob": float(probs[j]),
            "post_pred": int(pred[j]),
        })

    del clf, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

detector_summary = pd.DataFrame(summary_rows)
post_scores = pd.DataFrame(post_rows)

display(detector_summary)

detector_summary.to_csv(
    OUT/"detector_reproduction_summary.csv",
    index=False,
)

post_scores.to_parquet(
    OUT/"external_detector_post.parquet",
    index=False,
)



== openai_base ==


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 10728.62it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
openai_base post-ALISON: 100%|██████████████████████████████████████████████████████████████████████| 61/61 [00:25<00:00,  2.38batch/s]


retained n: 965
post accuracy: 0.5730569948186528 | paper: 0.5
post F1: 0.23985239852398524 | paper: 0.367

== openai_large ==


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 393/393 [00:00<00:00, 10267.35it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-large-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generic checkpoint labels: {0: 'LABEL_0', 1: 'LABEL_1'} | using class 0 = Fake/generated


openai_large post-ALISON: 100%|█████████████████████████████████████████████████████████████████████| 56/56 [01:14<00:00,  1.33s/batch]


retained n: 882
post accuracy: 0.608843537414966 | paper: 0.5682
post F1: 0.24836601307189543 | paper: 0.3623


,detector,n_retained,baseline_accuracy,baseline_f1,post_accuracy,paper_accuracy,post_f1,paper_f1,accuracy_abs_error,f1_abs_error
0,openai_base,965,1.0,1.0,0.573057,0.5000,0.239852,0.3670,0.073057,0.127148
1,openai_large,882,1.0,1.0,0.608844,0.5682,0.248366,0.3623,0.040644,0.113934



## 11. Semantic preservation

Compute METEOR and BERTScore once per transformed text, then report the mean over each detector's originally-correct retained subset, matching the paper's evaluation protocol.

We omit USE from the initial gate to avoid bringing TensorFlow/TF-Hub into the RELIC environment. The paper's BERTScore target remains the principal semantic reproduction landmark.


In [15]:

from bert_score import score as bertscore

SEM_PATH = OUT / "semantic_metrics.parquet"

if SEM_PATH.exists():
    semantic = pd.read_parquet(SEM_PATH)

else:
    semantic = obf[
        ["source_id","original","obfuscated_release"]
    ].copy()

    semantic["meteor"] = [
        meteor_score(
            [word_tokenize(original)],
            word_tokenize(transformed),
        )
        for original, transformed in tqdm(
            zip(
                semantic.original,
                semantic.obfuscated_release,
            ),
            total=len(semantic),
            desc="METEOR",
            unit="text",
        )
    ]

    P, R, F1 = bertscore(
        semantic.obfuscated_release.tolist(),
        semantic.original.tolist(),
        lang="en",
        verbose=True,
        device=DEVICE,
        batch_size=16,
    )

    semantic["bertscore_f1"] = F1.cpu().numpy()
    semantic.to_parquet(SEM_PATH, index=False)

sem_rows = []

for detector_name in DETECTORS:
    retained_ids = set(
        baseline_scores[
            baseline_scores.detector.eq(detector_name)
            & baseline_scores.original_correct
        ].source_id.astype(str)
    )

    s = semantic[
        semantic.source_id.astype(str).isin(retained_ids)
    ]

    target = PAPER_TARGETS[detector_name]

    sem_rows.append({
        "detector": detector_name,
        "n_retained": len(s),
        "meteor": float(s.meteor.mean()),
        "paper_meteor": target["meteor"],
        "bertscore": float(s.bertscore_f1.mean()),
        "paper_bertscore": target["bertscore"],
    })

semantic_summary = pd.DataFrame(sem_rows)
display(semantic_summary)

semantic_summary.to_csv(
    OUT/"semantic_reproduction_summary.csv",
    index=False,
)


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 11725.78it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|████████████████████████████████████████████████████████████████████████████████████████████████| 123/123 [01:30<00:00,  1.36it/s]


computing greedy matching.


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:02<00:00, 25.20it/s]


done in 93.17 seconds, 10.73 sentences/sec


,detector,n_retained,meteor,paper_meteor,bertscore,paper_bertscore
0,openai_base,965,0.834695,0.6966,0.926975,0.8941
1,openai_large,882,0.835488,0.6948,0.926544,0.9017



## 12. Intervention audit for later RELIC work

Paper reproduction is only the first goal. We also explicitly characterize ALISON as an intervention:

- number of POS-pattern matches;
- number of changed spans;
- replaced-token fraction;
- runtime;
- transformed Human vs transformed ChatGPT separately;
- release-lowercased vs case-preserved outputs.

This is the dataset C03 will use to ask whether \(U\), \(F\), and \(R\) survive or move under a validated local stylometric obfuscator.


In [16]:

audit = (
    obf
    .groupby("class_name")[
        [
            "n_candidate_intervals",
            "n_changed_intervals",
            "n_replaced_tokens",
            "n_lexically_changed_tokens",
            "masked_word_fraction",
            "lexical_change_fraction",
            "replacement_failures",
            "seconds",
        ]
    ]
    .agg(["mean","std","median"])
)

display(audit)

with open(OUT/"metadata.json", "w") as f:
    json.dump({
        "alison_current_commit": commit,
        "historical_functional_nn_commit": "e47d009eda89bd42a13d49c32ebc9c1e56b4d569",
        "chatgpt_parser": "paired multiline sections: label 0 Human, label 1 ChatGPT",
        "human_source": "label-0 blocks in ALISON/Data/ChatGPT.txt",
        "binary_labels": {"human":0, "chatgpt":1},
        "pair_aware_sampling_and_split": True,
        "n_per_class": N_PER_CLASS,
        "internal_fraction": INTERNAL_FRACTION,
        "V": V,
        "top_ngrams_per_length_max": TOP_NGRAMS,
        "top_pos_patterns": TOP_POS_PATTERNS,
        "c_length": C_LENGTH,
        "min_pos_length_exclusive": MIN_POS_LENGTH_EXCLUSIVE,
        "ig_steps": IG_STEPS,
        "paper_reproduction_output": "obfuscated_release",
        "relic_control_output": "obfuscated_case_preserved",
        "compatibility_fixes": [
            "ChatGPT.txt parsed as paired multiline label-0 Human and label-1 ChatGPT sections",
            "sampling and train/attack assignment grouped by matched pair_id",
            "feature-count assertion changed from exact 256 to <=256 per n",
            "historical functional width-512 ALISON MLP restored from earlier repository revision",
            "feature basis uses paper-specified character+POS ngrams only",
            "training StandardScaler reused for attack texts",
            "IG ranking uses scaled true-class POS attributions",
            "undefined ranked_indexes/obfuscated_texts release defects repaired",
            "BERT replacement uses actual tokenizer mask positions",
            "masked/touched and true lexical-change fractions are reported separately",
            "case-preserved edit output saved separately from release lowercasing",
        ],
        "paper_targets": PAPER_TARGETS,
    }, f, indent=2)

print("saved metadata:", OUT/"metadata.json")


n_candidate_intervals                   n_changed_intervals  \
                            mean        std median                mean   
class_name                                                               
chatgpt                   34.944   8.847226   33.0               19.88   
human                     47.016  28.968571   38.0              28.482   

                            n_replaced_tokens                    \
                 std median              mean        std median   
class_name                                                        
chatgpt     3.973560   20.0            67.762  11.959574   67.0   
human       9.861449   27.0            81.122  26.129996   79.0   

           n_lexically_changed_tokens  ... masked_word_fraction  \
                                 mean  ...               median   
class_name                             ...                        
chatgpt                        39.324  ...             0.267779   
human                          47.618  ...             0.209904   

           lexical_change_fraction                     replacement_failures  \
                              mean       std    median                 mean   
class_name                                                                    
chatgpt                   0.157307  0.032439  0.155102                  0.0   
human                     0.129289  0.038735  0.125455                  0.0   

                         seconds                      
            std median      mean       std    median  
class_name                                            
chatgpt     0.0    0.0  1.096339  0.239666  1.096905  
human       0.0    0.0  1.861398  0.725288  1.748797  

[2 rows x 24 columns]

saved metadata: ../results/causal/C02b_alison_reproduction_v2/metadata.json



## 13. Manual before/after audit


In [17]:

for r in obf.sample(min(8, len(obf)), random_state=SEED).itertuples(index=False):
    print("="*100)
    print(
        r.class_name,
        "| changed spans", r.n_changed_intervals,
        "| masked frac", round(float(r.masked_word_fraction), 3),
        "| lexical frac", round(float(r.lexical_change_fraction), 3),
        "| sec", round(float(r.seconds), 3),
    )

    print("\nORIGINAL\n", r.original[:2200])
    print("\nALISON — RELEASE\n", r.obfuscated_release[:2200])
    print("\nALISON — CASE PRESERVED\n", r.obfuscated_case_preserved[:2200])
    print()


chatgpt | changed spans 21 | masked frac 0.253 | lexical frac 0.173 | sec 1.056

ORIGINAL
 Hundreds of topless protesters took to the streets today, determined to make a statement in support of their right to tan. The protesters, who ranged in age and gender, marched topless through the city center, chanting and carrying signs with slogans like "Free the Tan" and "Our Bodies, Our Choice." The rally was organized in response to a recent proposed law that would require women to cover up from the waist up in public spaces. Participants said the proposed law would be oppressive and discriminatory, and that it would interfere with their right to do what they please with their own bodies. Though some spectators expressed shock at the sight of topless protesters, most seemed to show approval and understanding. One spectator, Maria Gomez, said, "I think this is a powerful statement for women's rights. We should not have to cover up in order to feel comfortable in our own skin." The march ended


## 14. Artifact integrity check

The expensive assets are the corrected split, POS cache, feature matrix, trained internal model, IG rankings, and BERT-replaced outputs. Detector and semantic summaries can be recomputed.


In [18]:

checks = {
    "binary_split": OUT/"binary_split.parquet",
    "pos_cache": OUT/"pos_cache.parquet",
    "features": OUT/"features.pkl",
    "feature_matrix": OUT/"feature_matrix.npy",
    "scaler": OUT/"scaler.pkl",
    "internal_model": OUT/"internal_mlp_historical.pt",
    "detector_baseline": OUT/"external_detector_baseline.parquet",
    "ig_rankings": OUT/"ig_ranked_pos.parquet",
    "alison_outputs": OUT/"alison_outputs.parquet",
    "detector_summary": OUT/"detector_reproduction_summary.csv",
    "semantic_metrics": OUT/"semantic_metrics.parquet",
    "metadata": OUT/"metadata.json",
}

for name, path in checks.items():
    print(
        f"{name:20s}",
        "OK" if path.exists() else "MISSING",
        "|",
        path,
    )

assert checks["alison_outputs"].exists()

saved = pd.read_parquet(
    checks["alison_outputs"]
)

expected_attack = int(
    (data.split == "attack_test").sum()
)

assert len(saved) == expected_attack
assert not saved.source_id.duplicated().any()
assert saved.obfuscated_release.notna().all()
assert saved.obfuscated_release.astype(str).str.strip().ne("").all()
assert saved.obfuscated_case_preserved.notna().all()
assert saved.obfuscated_case_preserved.astype(str).str.strip().ne("").all()
assert saved["source_id"].astype(str).isin(
    data.loc[data.split.eq("attack_test"), "source_id"].astype(str)
).all()

print("\nPASS: C02b v2 artifacts are complete and internally consistent.")


binary_split         OK | ../results/causal/C02b_alison_reproduction_v2/binary_split.parquet
pos_cache            OK | ../results/causal/C02b_alison_reproduction_v2/pos_cache.parquet
features             OK | ../results/causal/C02b_alison_reproduction_v2/features.pkl
feature_matrix       OK | ../results/causal/C02b_alison_reproduction_v2/feature_matrix.npy
scaler               OK | ../results/causal/C02b_alison_reproduction_v2/scaler.pkl
internal_model       OK | ../results/causal/C02b_alison_reproduction_v2/internal_mlp_historical.pt
detector_baseline    OK | ../results/causal/C02b_alison_reproduction_v2/external_detector_baseline.parquet
ig_rankings          OK | ../results/causal/C02b_alison_reproduction_v2/ig_ranked_pos.parquet
alison_outputs       OK | ../results/causal/C02b_alison_reproduction_v2/alison_outputs.parquet
detector_summary     OK | ../results/causal/C02b_alison_reproduction_v2/detector_reproduction_summary.csv
semantic_metrics     OK | ../results/causal/C02b_alison_r

In [19]:
# ============================================================
# C02b — retained-subset class behavior and F1 audit
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)

# Reload if this is a fresh kernel.
BASELINE_PATH = (
    OUT
    / "external_detector_baseline.parquet"
)

POST_PATH = (
    OUT
    / "external_detector_post.parquet"
)

baseline_scores = pd.read_parquet(
    BASELINE_PATH
)

post_scores = pd.read_parquet(
    POST_PATH
)

audit = (
    baseline_scores[
        baseline_scores["original_correct"]
    ]
    .merge(
        post_scores,
        on=[
            "source_id",
            "detector",
            "label",
        ],
        how="inner",
        validate="one_to_one",
    )
)

rows = []

for detector, d in audit.groupby(
    "detector",
    sort=False,
):
    y = d["label"].astype(int)
    p = d["post_pred"].astype(int)

    cm = confusion_matrix(
        y,
        p,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "detector": detector,
        "n_retained": len(d),

        "retained_human": int(
            (y == 0).sum()
        ),
        "retained_chatgpt": int(
            (y == 1).sum()
        ),

        "human_still_human_rate":
            tn / max(1, tn + fp),

        "chatgpt_still_detected_rate":
            tp / max(1, tp + fn),

        "chatgpt_evasion_rate":
            fn / max(1, tp + fn),

        "accuracy":
            accuracy_score(y, p),

        "balanced_accuracy":
            balanced_accuracy_score(y, p),

        "f1_binary_chatgpt":
            f1_score(
                y,
                p,
                average="binary",
                pos_label=1,
                zero_division=0,
            ),

        "f1_macro":
            f1_score(
                y,
                p,
                average="macro",
                zero_division=0,
            ),

        "f1_weighted":
            f1_score(
                y,
                p,
                average="weighted",
                zero_division=0,
            ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

audit_summary = pd.DataFrame(rows)

display(audit_summary)

print("\nClass-wise prediction counts:")

display(
    audit.groupby(
        [
            "detector",
            "class_name",
            "post_pred",
        ]
    )
    .size()
    .rename("n")
    .to_frame()
)

audit_summary.to_csv(
    OUT / "detector_classwise_audit.csv",
    index=False,
)

print(
    "\nsaved:",
    OUT / "detector_classwise_audit.csv",
)

,detector,n_retained,retained_human,retained_chatgpt,human_still_human_rate,chatgpt_still_detected_rate,chatgpt_evasion_rate,accuracy,balanced_accuracy,f1_binary_chatgpt,f1_macro,f1_weighted,tn,fp,fn,tp
0,openai_base,965,497,468,0.981891,0.138889,0.861111,0.573057,0.560390,0.239852,0.471511,0.478473,488,9,403,65
1,openai_large,882,500,382,0.960000,0.149215,0.850785,0.608844,0.554607,0.248366,0.491999,0.524594,480,20,325,57



Class-wise prediction counts:


n
detector     class_name post_pred     
openai_base  chatgpt    0          403
                        1           65
             human      0          488
                        1            9
openai_large chatgpt    0          325
                        1           57
             human      0          480
                        1           20


saved: ../results/causal/C02b_alison_reproduction_v2/detector_classwise_audit.csv


In [20]:
# ============================================================
# C02b — matched-pair 95% CIs:
# Human vs ChatGPT vs ALISON-reconstructed ChatGPT
# ============================================================

import gc
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)


# ------------------------------------------------------------
# Paths and configuration
# ------------------------------------------------------------

SPLIT_PATH = OUT / "binary_split.parquet"
BASELINE_PATH = OUT / "external_detector_baseline.parquet"
OBF_PATH = OUT / "alison_outputs.parquet"

POST_ALL_PATH = (
    OUT
    / "external_detector_post_all.parquet"
)

CI_PATH = (
    OUT
    / "detector_group_confidence_intervals.csv"
)

DELTA_CI_PATH = (
    OUT
    / "detector_pairwise_delta_confidence_intervals.csv"
)

DETECTORS = {
    "openai_base":
        "openai-community/roberta-base-openai-detector",

    "openai_large":
        "openai-community/roberta-large-openai-detector",
}

BATCH_SIZE = 16
N_BOOT = 10_000
CI_LEVEL = 0.95
CI_SEED = 42


# ------------------------------------------------------------
# Assemble the full, unbiased attack population
# ------------------------------------------------------------

data = pd.read_parquet(
    SPLIT_PATH
)

baseline_scores = pd.read_parquet(
    BASELINE_PATH
)

obf = pd.read_parquet(
    OBF_PATH
)

attack = (
    data[
        data["split"].eq("attack_test")
    ][
        [
            "pair_id",
            "source_id",
            "label",
            "class_name",
        ]
    ]
    .merge(
        obf[
            [
                "source_id",
                "obfuscated_release",
                "obfuscated_case_preserved",
            ]
        ],
        on="source_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        [
            "pair_id",
            "label",
        ]
    )
    .reset_index(drop=True)
)

assert len(attack) == 1000
assert attack.groupby("pair_id").size().eq(2).all()
assert attack.groupby("pair_id").label.nunique().eq(2).all()


# ------------------------------------------------------------
# Score every transformed sample
#
# Earlier post_scores contained only detector-specific
# originally-correct subsets. That is correct for reproducing
# the paper, but inappropriate for an unbiased group CI.
# ------------------------------------------------------------

if POST_ALL_PATH.exists():

    post_all = pd.read_parquet(
        POST_ALL_PATH
    )

    print(
        "loaded cached full post-ALISON scores:",
        POST_ALL_PATH,
    )

else:

    post_rows = []

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    for detector_name, checkpoint in DETECTORS.items():

        print(
            "\n==",
            detector_name,
            "==",
        )

        tok = AutoTokenizer.from_pretrained(
            checkpoint
        )

        clf = (
            AutoModelForSequenceClassification
            .from_pretrained(checkpoint)
            .to(DEVICE)
            .eval()
        )

        id2label = {
            int(k): str(v)
            for k, v
            in clf.config.id2label.items()
        }

        fake_matches = [
            k
            for k, v in id2label.items()
            if "fake" in v.lower()
        ]

        if fake_matches:
            fake_id = fake_matches[0]

        else:
            assert checkpoint in set(
                DETECTORS.values()
            )
            assert clf.config.num_labels == 2

            # Released OpenAI detector convention:
            # 0 = Fake/generated, 1 = Real/human.
            fake_id = 0

            print(
                "generic labels:",
                id2label,
                "| class 0 = Fake/generated",
            )

        probs = []

        with torch.inference_mode():

            for start in tqdm(
                range(
                    0,
                    len(attack),
                    BATCH_SIZE,
                ),
                desc=f"{detector_name} full ALISON set",
                unit="batch",
            ):

                batch = (
                    attack[
                        "obfuscated_release"
                    ]
                    .iloc[
                        start:
                        start + BATCH_SIZE
                    ]
                    .tolist()
                )

                enc = tok(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt",
                ).to(DEVICE)

                p = (
                    clf(**enc)
                    .logits
                    .softmax(-1)[:, fake_id]
                    .cpu()
                    .numpy()
                )

                probs.extend(
                    p.tolist()
                )

        assert len(probs) == len(attack)

        for source_id, prob in zip(
            attack["source_id"],
            probs,
        ):
            post_rows.append({
                "source_id":
                    str(source_id),

                "detector":
                    detector_name,

                "alison_fake_prob":
                    float(prob),
            })

        del clf, tok

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    post_all = pd.DataFrame(
        post_rows
    )

    post_all.to_parquet(
        POST_ALL_PATH,
        index=False,
    )

    print(
        "\nsaved:",
        POST_ALL_PATH,
    )


# ------------------------------------------------------------
# Combine original and transformed probabilities
# ------------------------------------------------------------

scores = (
    baseline_scores[
        [
            "source_id",
            "detector",
            "original_fake_prob",
        ]
    ]
    .merge(
        post_all,
        on=[
            "source_id",
            "detector",
        ],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        attack[
            [
                "pair_id",
                "source_id",
                "label",
                "class_name",
            ]
        ],
        on="source_id",
        how="inner",
        validate="many_to_one",
    )
)

assert len(scores) == (
    len(attack)
    * len(DETECTORS)
)


# ------------------------------------------------------------
# Matched-pair bootstrap
# ------------------------------------------------------------

alpha = (
    1.0
    - CI_LEVEL
) / 2.0


def summarize_bootstrap(
    values,
    boot_indices,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    boot = (
        values[boot_indices]
        .mean(axis=1)
    )

    lo, hi = np.quantile(
        boot,
        [
            alpha,
            1.0 - alpha,
        ],
    )

    return (
        float(values.mean()),
        float(lo),
        float(hi),
    )


ci_rows = []
delta_rows = []


for detector_name in DETECTORS:

    d = scores[
        scores["detector"].eq(
            detector_name
        )
    ]

    original = d.pivot(
        index="pair_id",
        columns="class_name",
        values="original_fake_prob",
    )

    transformed = d.pivot(
        index="pair_id",
        columns="class_name",
        values="alison_fake_prob",
    )

    paired = pd.DataFrame({
        "human":
            original["human"],

        "chatgpt":
            original["chatgpt"],

        "alison_human":
            transformed["human"],

        "alison_chatgpt":
            transformed["chatgpt"],
    }).dropna()

    assert len(paired) == 500

    rng = np.random.default_rng(
        CI_SEED
    )

    boot_indices = rng.integers(
        0,
        len(paired),
        size=(
            N_BOOT,
            len(paired),
        ),
    )

    groups = {
        "Human H":
            paired["human"].to_numpy(),

        "ChatGPT A":
            paired["chatgpt"].to_numpy(),

        "ALISON Human AL(H)":
            paired["alison_human"].to_numpy(),

        "ALISON ChatGPT AL(A)":
            paired["alison_chatgpt"].to_numpy(),
    }

    for group_name, values in groups.items():

        (
            mean_prob,
            prob_lo,
            prob_hi,
        ) = summarize_bootstrap(
            values,
            boot_indices,
        )

        predicted_fake = (
            values >= 0.5
        ).astype(float)

        (
            fake_rate,
            rate_lo,
            rate_hi,
        ) = summarize_bootstrap(
            predicted_fake,
            boot_indices,
        )

        ci_rows.append({
            "detector":
                detector_name,

            "group":
                group_name,

            "n_pairs":
                len(paired),

            "mean_fake_probability":
                mean_prob,

            "fake_probability_ci_low":
                prob_lo,

            "fake_probability_ci_high":
                prob_hi,

            "predicted_fake_rate":
                fake_rate,

            "predicted_fake_rate_ci_low":
                rate_lo,

            "predicted_fake_rate_ci_high":
                rate_hi,
        })

    pairwise_deltas = {
        "A - H":
            (
                paired["chatgpt"]
                - paired["human"]
            ).to_numpy(),

        "AL(A) - H":
            (
                paired["alison_chatgpt"]
                - paired["human"]
            ).to_numpy(),

        "A - AL(A)":
            (
                paired["chatgpt"]
                - paired["alison_chatgpt"]
            ).to_numpy(),

        "AL(H) - H":
            (
                paired["alison_human"]
                - paired["human"]
            ).to_numpy(),

        "AL(A) - AL(H)":
            (
                paired["alison_chatgpt"]
                - paired["alison_human"]
            ).to_numpy(),
    }

    for contrast, values in pairwise_deltas.items():

        (
            mean_delta,
            delta_lo,
            delta_hi,
        ) = summarize_bootstrap(
            values,
            boot_indices,
        )

        delta_rows.append({
            "detector":
                detector_name,

            "contrast":
                contrast,

            "n_pairs":
                len(paired),

            "mean_fake_probability_delta":
                mean_delta,

            "ci_low":
                delta_lo,

            "ci_high":
                delta_hi,

            "ci_excludes_zero":
                bool(
                    delta_lo > 0
                    or delta_hi < 0
                ),
        })


group_ci = pd.DataFrame(
    ci_rows
)

delta_ci = pd.DataFrame(
    delta_rows
)


# ------------------------------------------------------------
# Display and save
# ------------------------------------------------------------

print(
    "\n95% matched-pair bootstrap CIs "
    "for detector fake probability:"
)

display(
    group_ci.round(4)
)

print(
    "\n95% matched-pair bootstrap CIs "
    "for probability differences:"
)

display(
    delta_ci.round(4)
)

group_ci.to_csv(
    CI_PATH,
    index=False,
)

delta_ci.to_csv(
    DELTA_CI_PATH,
    index=False,
)

print(
    "\nsaved:",
    CI_PATH,
)

print(
    "saved:",
    DELTA_CI_PATH,
)


== openai_base ==


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 10765.47it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
openai_base full ALISON set: 100%|██████████████████████████████████████████████████████████████████| 63/63 [00:25<00:00,  2.46batch/s]



== openai_large ==


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 393/393 [00:00<00:00, 11174.80it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-large-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generic labels: {0: 'LABEL_0', 1: 'LABEL_1'} | class 0 = Fake/generated


...


saved: ../results/causal/C02b_alison_reproduction_v2/external_detector_post_all.parquet

95% matched-pair bootstrap CIs for detector fake probability:


,detector,group,n_pairs,mean_fake_probability,fake_probability_ci_low,fake_probability_ci_high,predicted_fake_rate,predicted_fake_rate_ci_low,predicted_fake_rate_ci_high
0,openai_base,Human H,500,0.0127,0.0066,0.0206,0.006,0.000,0.014
1,openai_base,ChatGPT A,500,0.9374,0.9176,0.9555,0.936,0.914,0.956
2,openai_base,ALISON Human AL(H),500,0.0202,0.0119,0.0300,0.018,0.008,0.030
3,openai_base,ALISON ChatGPT AL(A),500,0.1402,0.1134,0.1686,0.132,0.104,0.162
4,openai_large,Human H,500,0.0080,0.0054,0.0112,0.000,0.000,0.000
5,openai_large,ChatGPT A,500,0.7405,0.7107,0.7699,0.764,0.726,0.800
6,openai_large,ALISON Human AL(H),500,0.0793,0.0651,0.0947,0.040,0.024,0.058
7,openai_large,ALISON ChatGPT AL(A),500,0.1632,0.1405,0.1867,0.132,0.102,0.162



95% matched-pair bootstrap CIs for probability differences:


,detector,contrast,n_pairs,mean_fake_probability_delta,ci_low,ci_high,ci_excludes_zero
0,openai_base,A - H,500,0.9247,0.9035,0.9441,True
1,openai_base,AL(A) - H,500,0.1275,0.0999,0.1563,True
2,openai_base,A - AL(A),500,0.7972,0.7653,0.8282,True
3,openai_base,AL(H) - H,500,0.0075,-0.0041,0.0192,False
4,openai_base,AL(A) - AL(H),500,0.1200,0.0928,0.1483,True
5,openai_large,A - H,500,0.7325,0.7026,0.7619,True
6,openai_large,AL(A) - H,500,0.1552,0.1325,0.1789,True
7,openai_large,A - AL(A),500,0.5773,0.5449,0.6102,True
8,openai_large,AL(H) - H,500,0.0713,0.0567,0.0867,True
9,openai_large,AL(A) - AL(H),500,0.0839,0.0577,0.1102,True



saved: ../results/causal/C02b_alison_reproduction_v2/detector_group_confidence_intervals.csv
saved: ../results/causal/C02b_alison_reproduction_v2/detector_pairwise_delta_confidence_intervals.csv
